# Symmetry And Fundamental Regions

Symmetry is where many texture workflows become ambiguous in other tools. PyTex
treats `SymmetrySpec` as a reusable scientific object and exposes explicit
vector- and orientation-reduction paths.

## Mathematical Idea

A symmetry orbit acts on a direction or orientation by group multiplication. A
reduction then selects one representative from that orbit according to an explicit
rule rather than a hidden plotting convention.


In [ ]:
import numpy as np

from pytex import (
    AcquisitionGeometry,
    BenchmarkManifest,
    CalibrationRecord,
    CrystalMap,
    CrystalPlane,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    ReferenceFrame,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
)


def make_context():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
    lattice = Lattice(3.6, 3.6, 3.6, 90.0, 90.0, 90.0, crystal_frame=crystal)
    phase = Phase("fcc_demo", lattice=lattice, symmetry=symmetry, crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


In [ ]:
crystal, specimen, *_ = make_context()
symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)

directions = VectorSet(
    values=np.array(
        [
            [1.0, 1.0, 1.0],
            [-1.0, 1.0, 1.0],
            [1.0, -1.0, 1.0],
            [1.0, 1.0, -1.0],
        ]
    ),
    reference_frame=crystal,
).normalized()

reduced = symmetry.reduce_vectors_to_fundamental_sector(directions, antipodal=True)
print(reduced.values)


In [ ]:
orientations = OrientationSet.from_euler_angles(
    np.array(
        [
            [0.0, 0.0, 0.0],
            [90.0, 0.0, 0.0],
            [35.0, 20.0, 10.0],
        ]
    ),
    crystal_frame=crystal,
    specimen_frame=specimen,
    symmetry=symmetry,
)

projected = orientations.projected_to_fundamental_region()
keys = projected.exact_fundamental_region_keys()
print(keys)


The combination of `SymmetrySpec` plus explicit batch reduction is what lets PyTex
explain exactly how an inverse pole figure sector or orientation fundamental region
was obtained.
